# 环境配置

本节将完成 torchtitan-npu 的环境配置，确保你能在 Ascend NPU 上跑通 SFT 训练。

---

## 前置条件

在开始之前，确保以下软件已安装并可用：

| 组件 | 版本 | 验证命令 |
|------|------|--------|
| CANN | 9.0.0 | `npu-smi info` |
| PyTorch | 2.12.0 | `pip list \| grep torch` |
| torch_npu | 2.12.0rc1 | `pip list \| grep torch_npu` |

> **注意**：本教程假设 CANN 和 PyTorch 环境已由集群管理员配置好（`npu-smi info` 和 `import torch_npu` 均可正常执行）。如果不可用，请先参考 torchtitan-npu 官方的[安装指南](https://gitcode.com/cann/torchtitan-npu/blob/master/docs/user-guides/installation.md)完成 NPU 基础环境配置。

---

## 1. 克隆仓库

本教程使用 torchtitan-npu 的 `master` 分支。

> **目录提醒**：后续章节（例如 `04.03`）需要直接修改 torchtitan-npu 的源代码，建议将仓库放在可写的 workspace 内（例如 `/mnt/workspace/torchtitan-npu`），不要放在临时目录或只读目录中。


In [ ]:
%%bash
set -euo pipefail

# 可以用这个变量自定义下载路径。
repo=/mnt/workspace/torchtitan-npu
git clone  https://gitcode.com/cann/torchtitan-npu.git "$repo"
cd "$repo"
branch=$(git branch --show-current)
echo "Repo: $repo"
echo "Branch: $branch"


In [ ]:
import os
original_dir = os.getcwd()
%cd /mnt/workspace/torchtitan-npu
%env BASH_ENV=/home/developer/Ascend/cann/set_env.sh


---

## 2. 安装依赖


In [ ]:
%%bash
pip install -r requirements.txt
pip install -e .
pip list | grep -E 'torch |torch_npu |torchtitan|triton-ascend|safetensors'
pip install --user --force-reinstall --no-deps \
  "pyarrow==21.0.0"
pip install -U uv openai mcp httpx tenacity textarena==0.7.4
export UV_CACHE_DIR=/tmp/uv-cache
mkdir -p "$UV_CACHE_DIR"



`requirements.txt` 中包含：

| 依赖 | 版本 | 用途 |
|------|------|------|
| `torch` | 2.12.0+cpu | PyTorch 基础框架 |
| `torch_npu` | 2.12.0rc1 | Ascend NPU 后端支持 |
| `torchtitan` | git (ac13e53) | 训练框架核心 |
| `triton-ascend` | 3.2.1 | Ascend Triton 加速库 |
| `safetensors` | 0.7.0 | 权重文件安全加载 |

---

## 3. 验证环境：跑 Debug 模型

torchtitan-npu 提供了 debug 模式，用极小模型快速验证环境是否正常：


In [ ]:
%%bash
NGPU=2 bash scripts/run_train.sh



Debug 模式的特点：
- 使用极小模型（hidden_size 很小，只有几层），几秒内完成一个 step。
- 不依赖提前下载的权重（随机初始化）。
- 只验证数据加载、模型构建、前向/反向传播、梯度更新是否正常。

**预期输出**：


```text
Running with configs: model and recipe resolved from the current master defaults
visible dies: 2
step: 1  loss: <finite>  grad_norm: <finite>
Training completed
```

下载基座 Qwen3-1.7B 模型（约需 15 分钟）：


In [ ]:
%%bash
set -euo pipefail

MODEL_DIR="./assets/hf/Qwen3-1.7B"
python3 -m pip install -U modelscope
modelscope download \
  --model PrimeIntellect/Qwen3-1.7B \
  --local_dir "$MODEL_DIR"

pwd
ls ./assets/hf/Qwen3-1.7B


---

## 5. 下载数据集

Wordle SFT 训练使用 `willcb/V3-wordle` 数据集，包含 5 字母单词的猜词多轮对话：


In [ ]:
%%bash
# 1. 下载数据集（源文件已经是 Parquet 格式）
HF_HUB_DISABLE_XET=1 HF_ENDPOINT=https://hf-mirror.com hf download willcb/V3-wordle data/train-00000-of-00001.parquet --repo-type=dataset --local-dir ./assets/data/wordle_raw

# 2. 复制到 torchtitan 约定的数据路径
mkdir -p ./assets/data/wordle
cp ./assets/data/wordle_raw/data/train-00000-of-00001.parquet ./assets/data/wordle


Wordle 评测环境依赖 NLTK 语料库（`words` 和 `averaged_perceptron_tagger`）。以下脚本从 NLTK 官方源下载并自动解压到 `$HOME/nltk_data`（约需 7 分钟）：


In [ ]:
%%bash
pip install nltk
python3 -c "
import nltk
nltk.download('words')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('averaged_perceptron_tagger')
"



---

## 7. SFT 配置速览

在进入第 3 章之前，先看一眼我们即将使用的 SFT 配置入口：

```bash
# 正式训练命令（第 3 章会详细讲解）
NGPU=2 \
MODULE=torchtitan_npu.models.qwen3 \
CONFIG=sft_qwen3_1_7b_wordle \
bash scripts/run_train.sh
# ...
```



> **注意**：配置细节会在第 3 章数据准备和训练运行时详细展开，这里只需要知道"一条命令就能跑 SFT"。

---

## 小结

- Clone `cann/torchtitan-npu` 并切换到 `master` 分支。
- `pip install -r requirements.txt && pip install -e .` 完成安装。
- 下载 Qwen3-1.7B 基座模型权重。
- 下载并解压 NLTK 语料库到 `$HOME/nltk_data`（Wordle 评测环境依赖）。
- 用 debug 模式验证环境：`NGPU=2 bash scripts/run_train.sh`。

下一章，我们将真正启动 Wordle SFT 训练。

1. （单选题）`scripts/run_train.sh` 中 `MODULE` 和 `CONFIG` 分别指定什么？
    A. 模型模块与训练配方
    B. 数据目录与输出目录
    C. rank 数与 die 编号
    D. 驱动与 CANN 版本

2. （多选题）安装 torchtitan-npu 需要哪些步骤？
    A. 克隆仓库（master 分支）
    B. pip install -r requirements.txt && pip install -e .
    C. 编译 CANN 源码
    D. 下载 Qwen3-1.7B 基座模型权重


In [ ]:
%cd $original_dir
!cat ./answer/02.03_answer.txt